### Pipeline Flow:

1. User provides mood text, genres and favorite movie
2. detect_mood classifies the mood from free text
3. search_movies fetches 20 candidates from TMDB
4. get_movie_details and get_summary enrich each candidate
5. build_query and rank_candidates rank by semantic similarity
6. In couple mode, fuse_vectors combines two user profiles
7. generate calls Groq LLM for recommendations with WHY explanations
8. get_trailer fetches YouTube trailer for each recommendation
9. get_playlist fetches Spotify soundtrack for each recommended movie
10. save_preferences stores user history for future sessions

In [ ]:
# Run all notebooks to load all functions
%run notebooks/01_tmdb_api.ipynb
%run notebooks/02_wikipedia.ipynb
%run notebooks/03_youtube.ipynb
%run notebooks/04_spotify.ipynb
%run notebooks/05_embeddings.ipynb
%run notebooks/06_query_builder.ipynb
%run notebooks/07_llm.ipynb
%run notebooks/08_fusion.ipynb
%run notebooks/09_mood_detector.ipynb
%run notebooks/11_feedback.ipynb

Pylance cannot see functions that are loaded dynamically at runtime. All functions marked as not defined. Ignore it.

In [ ]:
def recommend(user_id, mood_text, genres, favorite, actor,
              avoid, min_rating, decade, mode, user_b=None):

    # 1. Detect mood
    mood = detect_mood(mood_text) if mood_text else "Happy"

    # 2. Fetch candidates from TMDB
    candidates = search_movies(genres, min_rating=min_rating, decade=decade)

    # 3. Enrich with details + Wikipedia
    enriched = []
    for c in candidates:
        details = get_movie_details(c["id"])
        details["wiki"] = get_summary(details["title"], details["year"])
        enriched.append(details)
    candidates = enriched

    # 4. Build query + embed + rank
    query = build_query(mood, genres, favorite, actor, avoid)
    ranked = rank_candidates(query, candidates)

    # 5. Couple fusion if couple mode
    if mode == "couple" and user_b:
        alpha, beta = compute_weights(
            {"genres":genres,"mood":mood,"favorite":favorite}, user_b)
        vec_a = embed(query)
        vec_b = embed(build_query(
            user_b["mood"], user_b["genres"],
            user_b["favorite"],
            user_b.get("actor"),
            user_b.get("avoid")
        ))
        compat_pct, compat_label = compatibility_score(vec_a, vec_b)
        fused_vec = fuse_vectors(vec_a, vec_b, alpha, beta)
        ranked = rank_by_vector(fused_vec, candidates)

    # 6. Filter disliked movies
    disliked = get_disliked(user_id)
    ranked = [r for r in ranked if r["title"] not in disliked]

    # 7. LLM final recommendation
    top5 = ranked[:5]
    llm_output = generate(top5, query, mode=mode)

    # 8. Enrich with trailer + playlist
    for movie in top5[:3]:
        movie["trailer"] = get_trailer(movie["title"], movie["year"])
        movie["playlist"] = get_playlist(movie["title"], movie["year"])

    # 9. Save preferences
    save_preferences(user_id, mood, genres, favorite, avoid)

    return top5[:3], llm_output

In [ ]:
#Test
movies, llm_output = recommend(
    user_id="hanna",
    mood_text="I want something exciting",
    genres=["Action", "Sci-Fi"],
    favorite="Interstellar",
    actor=None,
    avoid=[],
    min_rating=7.0,
    decade=None,
    mode="solo"
)
print(llm_output)